# ⚡ Módulo 14 - Notebook 02: Optimizador Catalyst y Plan Ejecución

## ⚙️ Entendiendo el Motor de Optimización de Spark

**Libro:** Saliendo de lo Pandito  
**Módulo:** 14 - PySpark Optimización ETL Pipelines  
**Duración estimada:** 70 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Entender** cómo funciona Catalyst Optimizer  
✅ **Interpretar** planes lógicos y físicos  
✅ **Identificar** optimizaciones automáticas  
✅ **Usar** explain() para debugging  
✅ **Aplicar** técnicas de optimización manual

---

## 📋 Pre-requisitos

* ✅ Notebook 14_01 completado
* ✅ Conocimiento de DataFrame API
* ✅ Familiaridad con SQL

---

## 📚 Contenido

1. ¿Qué es Catalyst?
2. Fases del Optimizador
3. Plan Lógico vs Físico
4. Optimizaciones Automáticas
5. explain() y Debugging
6. Caso Integrador: Optimizar Query Lenta

---

## 💡 Por qué importa

**Catalyst es el cerebro de Spark:**

* 🧠 **Inteligente:** Optimiza automáticamente
* 📊 **Transparente:** Mismo código, mejor performance
* 🔍 **Debuggeable:** explain() muestra el plan
* ⚡ **Rápido:** Optimizaciones probadas

**Entender Catalyst = escribir código más eficiente**

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market
    df = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros: {df.count():,}")
    print(f"   🗂️ Particiones: {df.rdd.getNumPartitions()}")
    
    # Ejemplo: Query compleja para analizar con Catalyst
    query_compleja = df \
        .filter(F.col("ventas") > 50000) \
        .groupBy("zona", "sucursal_nombre") \
        .agg(
            F.sum("ventas").alias("ventas_totales"),
            F.count("*").alias("transacciones")
        ) \
        .filter(F.col("ventas_totales") > 1000000) \
        .orderBy(F.desc("ventas_totales"))
    
    print(f"\n📊 Query de ejemplo creada (no ejecutada aún)")
    print(f"   Filtros: ventas > 50000, ventas_totales > 1000000")
    print(f"   Agregaciones: SUM(ventas), COUNT(*)")
    print(f"   Ordenamiento: DESC por ventas_totales")
    
    print(f"\n📋 PLAN LÓGICO (Simple):")
    query_compleja.explain(mode="simple")
    
    print(f"\n📋 PLAN EXTENDIDO (Todas las fases):")
    query_compleja.explain(mode="extended")
    
    print(f"\n🎯 Este notebook analizará:")
    print(f"   • Cómo Catalyst optimiza esta query")
    print(f"   • Predicate pushdown (filtrar temprano)")
    print(f"   • Projection pruning (leer solo columnas necesarias)")
    print(f"   • Join reordering (si aplicara)")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df = None
    query_compleja = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Catalyst: El Cerebro de Spark

### ⚙️ ¿Qué es Catalyst?

**Catalyst** es el optimizador de queries basado en reglas de Spark SQL.

**Función:**
Convertir tu código (DataFrame API o SQL) en el plan de ejecución más eficiente.

---

### 📄 Fases del Optimizador

**Flujo completo:**

```
1️⃣ Análisis (Analysis)
   Tu código → Árbol sintáctico abstracto (AST)
   Valida esquema, resuelve nombres de columnas
   ↓
2️⃣ Plan Lógico (Logical Plan)
   Qué operaciones hacer (sin detalles de implementación)
   ↓
3️⃣ Optimización Lógica
   Aplica reglas de optimización (predicate pushdown, etc.)
   ↓
4️⃣ Plan Físico (Physical Plan)
   Cómo ejecutar (con estrategias concretas)
   ↓
5️⃣ Generación de Código (Code Generation)
   Genera bytecode Java optimizado (Tungsten)
   ↓
6️⃣ Ejecución
   Ejecuta en cluster distribuido
```

---

### 📋 Tipos de Planes

#### 1️⃣ Plan Lógico

**Qué hacer** (sin detalles de implementación).

```python
df.filter("ventas > 100000").groupBy("zona").sum("ventas").explain(mode="simple")
```

**Salida:**
```
== Physical Plan ==
Project [zona, sum(ventas)]
+- HashAggregate [zona], [sum(ventas)]
   +- Filter (ventas > 100000)
      +- Scan parquet [zona, ventas]
```

---

#### 2️⃣ Plan Físico

**Cómo ejecutar** (estrategias concretas).

```python
df.filter("ventas > 100000").groupBy("zona").sum("ventas").explain(mode="formatted")
```

**Detalles:**
* Qué implementación de join usar (SortMerge, Broadcast, etc.)
* Cuántas particiones
* Tamaño estimado de datos

---

### 🚀 Optimizaciones Automáticas

#### 1️⃣ **Predicate Pushdown**

Mover filtros lo más cerca posible del origen.

**Sin optimización:**
```
1. Leer TODO el archivo Parquet
2. Filtrar ventas > 100000
```

**Con Predicate Pushdown:**
```
1. Leer SOLO registros con ventas > 100000 (desde Parquet)
```

**Beneficio:** Lee menos datos = más rápido

---

#### 2️⃣ **Projection Pruning**

Leer solo las columnas que necesitas.

**Sin optimización:**
```python
df.select("zona", "ventas")  # Lee TODO el archivo (100 columnas)
```

**Con Projection Pruning:**
```python
df.select("zona", "ventas")  # Lee SOLO zona y ventas
```

**Beneficio:** Menos I/O = más rápido

---

#### 3️⃣ **Constant Folding**

Evaluar expresiones constantes una sola vez.

**Tu código:**
```python
df.filter(F.col("ventas") > 1000 * 100)  # 1000 * 100 se evalúa por cada fila
```

**Catalyst lo optimiza a:**
```python
df.filter(F.col("ventas") > 100000)  # Evaluado una sola vez
```

---

#### 4️⃣ **Join Reordering**

Cambiar el orden de joins para minimizar datos intermedios.

**Tu código:**
```python
df1.join(df2, "id").join(df3, "id")
```

**Catalyst puede reordenar:**
```python
df1.join(df3, "id").join(df2, "id")  # Si df3 es más pequeño
```

---

### 🔍 explain() - Tu Herramienta de Debugging

**Modos:**

```python
# Simple: Solo plan físico
df.explain(mode="simple")

# Extended: Todos los planes (lógico, optimizado, físico)
df.explain(mode="extended")

# Formatted: Plan físico formateado (más legible)
df.explain(mode="formatted")

# Cost: Incluye costos estimados
df.explain(mode="cost")
```

**Cuándo usar:**
* Query lenta → `explain("extended")` para ver optimizaciones
* Join pesado → Verificar si usa Broadcast o SortMerge
* Shuffle inesperado → Identificar dónde ocurre

---

### 📊 Cómo Leer un Plan de Ejecución

**Ejemplo:**
```
== Physical Plan ==
*(2) HashAggregate(keys=[zona#123], functions=[sum(ventas#124)])
+- Exchange hashpartitioning(zona#123, 200)  ← SHUFFLE!
   +- *(1) HashAggregate(keys=[zona#123], functions=[partial_sum(ventas#124)])
      +- *(1) Project [zona#123, ventas#124]
         +- *(1) Filter (ventas#124 > 100000)
            +- *(1) FileScan parquet [zona#123, ventas#124]
                  Batched: true
                  PushedFilters: [IsNotNull(ventas), GreaterThan(ventas,100000)]
```

**Lectura (de abajo hacia arriba):**

1. `FileScan parquet` → Lee archivo Parquet
2. `PushedFilters` → **Predicate Pushdown aplicado** ✅
3. `Filter` → Filtro en memoria (por si hay datos no filtrados)
4. `Project` → **Projection Pruning** (solo zona, ventas) ✅
5. `HashAggregate` (partial) → Agregación local por partición
6. `Exchange hashpartitioning` → **SHUFFLE** (costoso) ⚠️
7. `HashAggregate` (final) → Agregación final

---

### ⚠️ Identificar Problemas

**Red Flags en el plan:**

🔴 **Exchange** → Shuffle (costoso)  
🔴 **SortMergeJoin en tabla grande** → Considerar Broadcast  
🔴 **Sin PushedFilters** → Filtro no pushdown (leerá TODO)  
🔴 **Múltiples Exchange seguidos** → Repartir antes  

---

### 🎯 Cómo Ayudar a Catalyst

**1️⃣ Filtrar temprano**
```python
# Catalyst optimiza mejor si filtras primero
df.filter("fecha > '2024-01-01'").groupBy("zona").sum("ventas")
```

**2️⃣ Proyectar solo necesario**
```python
# No hagas select(*) si solo necesitas 3 columnas
df.select("zona", "ventas", "fecha")
```

**3️⃣ Usar tipos correctos**
```python
# Evita cast() innecesarios (Catalyst no puede optimizar)
df.withColumn("fecha", F.to_date("fecha_str"))  # Mejor que cast()
```

**4️⃣ Broadcast explícito para tablas pequeñas**
```python
from pyspark.sql.functions import broadcast
df_large.join(broadcast(df_small), "key")  # Evita shuffle
```

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

print("⚙️ CATALYST OPTIMIZER Y PLANES DE EJECUCIÓN")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    print(f"Versión de Spark: {spark.version}")
except:
    print("⚠️  SparkSession no disponible")

print("\n🎯 En este notebook aprenderás:")
print("  • Fases de Catalyst (Analysis, Logical, Physical)")
print("  • Plan Lógico vs Plan Físico")
print("  • Optimizaciones automáticas (Predicate Pushdown, Projection Pruning)")
print("  • explain() - Modos simple, extended, formatted, cost")
print("  • Cómo leer e interpretar planes")

print("\n📖 Métodos clave:")
print("  - df.explain(mode='simple')     # Plan físico")
print("  - df.explain(mode='extended')   # Todos los planes")
print("  - df.explain(mode='formatted')  # Plan formateado")
print("  - df.explain(mode='cost')       # Con costos estimados")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')